In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


In [2]:
# 1. Đọc dữ liệu bằng pandas
df = pd.read_csv(Path("../data/house_price_clean.csv"))

# Xử lý cột phân loại location bằng one-hot encoding
df = pd.get_dummies(df, columns=["location"], drop_first=True)

# 2. Xử lý giá trị bị thiếu
df = df.fillna(0)

# 3. Chuyển thành NumPy array để cắt lát X, y
target_col = "price_million_vnd"

X = df.drop(columns=[target_col]).to_numpy(dtype=float)
y = df[[target_col]].to_numpy(dtype=float)

print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: (64447, 276)
y shape: (64447, 1)


In [3]:
np.random.seed(42)

indices = np.random.permutation(len(X))

split = int(0.8 * len(X))

train_idx = indices[:split]
test_idx = indices[split:]

X_train, y_train = X[train_idx], y[train_idx]
X_test, y_test = X[test_idx], y[test_idx]

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


Training samples: 51557
Testing samples: 12890


In [4]:
mean = X_train.mean(axis=0)
std = X_train.std(axis=0) + 1e-8

X_train = (X_train - mean) / std
X_test = (X_test - mean) / std

# Chuẩn hóa y để huấn luyện mạng Neural ổn định trong bài toán hồi quy
y_mean = y_train.mean()
y_std = y_train.std() + 1e-8

y_train_scaled = (y_train - y_mean) / y_std
y_test_scaled = (y_test - y_mean) / y_std


In [5]:
def relu(x):
    return np.maximum(0, x)


def relu_derivative(x):
    return (x > 0).astype(float)


In [6]:
def linear(z):
    return z


def linear_derivative(a):
    return np.ones_like(a)


In [7]:
# Step 7
np.random.seed(42)

input_dim = X.shape[1]

# Layer 1: input_dim -> 32
W1 = np.random.randn(input_dim, 32) * np.sqrt(2.0 / input_dim)
b1 = np.zeros((1, 32))

# Layer 2: 32 -> 16
W2 = np.random.randn(32, 16) * np.sqrt(2.0 / 32)
b2 = np.zeros((1, 16))

# Layer 3: 16 -> 8
W3 = np.random.randn(16, 8) * np.sqrt(2.0 / 16)
b3 = np.zeros((1, 8))

# Layer 4: 8 -> 4
W4 = np.random.randn(8, 4) * np.sqrt(2.0 / 8)
b4 = np.zeros((1, 4))

# Layer 5: 4 -> 1 (Output)
W5 = np.random.randn(4, 1) * np.sqrt(2.0 / 4)
b5 = np.zeros((1, 1))


In [8]:
# Step 8
def forward(X):
    # Layer 1
    z1 = X @ W1 + b1
    h1 = relu(z1)

    # Layer 2
    z2 = h1 @ W2 + b2
    h2 = relu(z2)

    # Layer 3
    z3 = h2 @ W3 + b3
    h3 = relu(z3)

    # Layer 4
    z4 = h3 @ W4 + b4
    h4 = relu(z4)

    # Layer 5 (Output Linear)
    z5 = h4 @ W5 + b5
    y_hat = linear(z5)

    cache = {
        "X": X,
        "z1": z1,
        "h1": h1,
        "z2": z2,
        "h2": h2,
        "z3": z3,
        "h3": h3,
        "z4": z4,
        "h4": h4,
        "z5": z5,
        "y_hat": y_hat,
    }
    return y_hat, cache


In [9]:
# Step 9
def mse_loss(y, y_hat):
    return np.mean((y - y_hat) ** 2)


In [10]:
# Step 10
def backward(y, cache):
    X = cache["X"]
    z1, h1 = cache["z1"], cache["h1"]
    z2, h2 = cache["z2"], cache["h2"]
    z3, h3 = cache["z3"], cache["h3"]
    z4, h4 = cache["z4"], cache["h4"]
    yhat = cache["y_hat"]

    n = len(X)

    # ---------------------------
    # Layer 5 (Output Layer)
    # ---------------------------
    dz5 = 2.0 * (yhat - y) / n
    dW5 = h4.T @ dz5
    db5 = np.sum(dz5, axis=0, keepdims=True)

    # ---------------------------
    # Layer 4
    # ---------------------------
    dh4 = dz5 @ W5.T
    dz4 = dh4 * relu_derivative(z4)
    dW4 = h3.T @ dz4
    db4 = np.sum(dz4, axis=0, keepdims=True)

    # ---------------------------
    # Layer 3
    # ---------------------------
    dh3 = dz4 @ W4.T
    dz3 = dh3 * relu_derivative(z3)
    dW3 = h2.T @ dz3
    db3 = np.sum(dz3, axis=0, keepdims=True)

    # ---------------------------
    # Layer 2
    # ---------------------------
    dh2 = dz3 @ W3.T
    dz2 = dh2 * relu_derivative(z2)
    dW2 = h1.T @ dz2
    db2 = np.sum(dz2, axis=0, keepdims=True)

    # ---------------------------
    # Layer 1
    # ---------------------------
    dh1 = dz2 @ W2.T
    dz1 = dh1 * relu_derivative(z1)
    dW1 = X.T @ dz1
    db1 = np.sum(dz1, axis=0, keepdims=True)

    gradients = {
        "dW1": dW1,
        "db1": db1,
        "dW2": dW2,
        "db2": db2,
        "dW3": dW3,
        "db3": db3,
        "dW4": dW4,
        "db4": db4,
        "dW5": dW5,
        "db5": db5,
    }

    return gradients


In [11]:
# # Step 11
# learning_rate = 0.01

# W1 -= learning_rate * gradients["dW1"]
# b1 -= learning_rate * gradients["db1"]
# W2 -= learning_rate * gradients["dW2"]
# b2 -= learning_rate * gradients["db2"]
# W3 -= learning_rate * gradients["dW3"]
# b3 -= learning_rate * gradients["db3"]


In [12]:
# Step 12
learning_rate = 0.01
epochs = 1000

for epoch in range(epochs):
    # Forward pass
    y_hat, cache = forward(X_train)

    # Compute loss
    loss = mse_loss(y_train_scaled, y_hat)

    # Backward pass
    gradients = backward(y_train_scaled, cache)

    # Update weights and biases
    W1 -= learning_rate * gradients["dW1"]
    b1 -= learning_rate * gradients["db1"]
    W2 -= learning_rate * gradients["dW2"]
    b2 -= learning_rate * gradients["db2"]
    W3 -= learning_rate * gradients["dW3"]
    b3 -= learning_rate * gradients["db3"]
    W4 -= learning_rate * gradients["dW4"]
    b4 -= learning_rate * gradients["db4"]
    W5 -= learning_rate * gradients["dW5"]
    b5 -= learning_rate * gradients["db5"]

    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch + 1}/{epochs}, Loss: {loss:.4f}")


Epoch 100/1000, Loss: 1.0111
Epoch 200/1000, Loss: 0.9957
Epoch 300/1000, Loss: 0.9866
Epoch 400/1000, Loss: 0.9760
Epoch 500/1000, Loss: 0.9630
Epoch 600/1000, Loss: 0.9430
Epoch 700/1000, Loss: 0.9213
Epoch 800/1000, Loss: 0.9023
Epoch 900/1000, Loss: 0.8907
Epoch 1000/1000, Loss: 0.8836


In [13]:
# Step 13
y_hat_scaled, _ = forward(X_test)

y_pred = y_hat_scaled * y_std + y_mean


In [14]:
# Step 14
mse = np.mean((y_pred - y_test) ** 2)
rmse = np.sqrt(mse)

print("MSE:", mse)
print("RMSE:", rmse)


MSE: 2.2564789450088558e+18
RMSE: 1502158095.877014


In [15]:
# Step 15
mae = np.mean(np.abs(y_pred - y_test))
ss_res = np.sum((y_test - y_pred) ** 2)
ss_tot = np.sum((y_test - y_mean) ** 2)
r2 = 1.0 - (ss_res / (ss_tot + 1e-8))


In [16]:
# giai thich ly thuyet
print("MAE:", mae)
print("R2-Score:", r2)


MAE: 31585039.761631895
R2-Score: -2351754048.6225863
